# 🚀 Day 02a: Caching — The #1 Performance Tool

---

## 🎯 What You'll Master Today
- Cache strategies: cache-aside, write-through, write-behind, read-through
- Redis vs Memcached
- Eviction policies (LRU, LFU, TTL)
- Cache invalidation — the hardest problem
- Cache stampede & thundering herd

**"There are only two hard things in CS: cache invalidation and naming things."**

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  CACHING LAYERS IN A SYSTEM                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Client ──► CDN ──► API Gateway ──► App Server ──► Cache ──► DB ║
║    │          │         │               │            │            ║
║  Browser    Edge     Rate limit      L1: In-proc   L2: Redis    ║
║  cache      cache    cache           (dict/LRU)    /Memcached   ║
║                                                                   ║
║  Every layer can cache! The question is WHERE and HOW LONG.      ║
║                                                                   ║
║  Cache hit ratio target: 90-99%                                  ║
║  Miss penalty: extra latency (cache check + DB query)            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Cache-Aside (Lazy Loading) — Most Common
# ============================================================

print("""
CACHE-ASIDE (Lazy Loading)
══════════════════════════

READ:                              WRITE:
  1. App checks cache               1. App writes to DB
  2. HIT → return from cache        2. App invalidates cache
  3. MISS → read from DB               (delete key)
  4. Store in cache + return

  App ──check──► Cache              App ──write──► DB
   │                │                │
   │   HIT: return  │                └─delete──► Cache
   │   MISS: ↓      │
   └──query──► DB
   └──store──► Cache
""")

class CacheAside:
    def __init__(self):
        self.cache = {} 
        self.db = {"user:1": "Alice", "user:2": "Bob", "user:3": "Charlie"}
        self.stats = {"hits": 0, "misses": 0}
    
    def read(self, key):
        # Step 1: Check cache
        if key in self.cache:
            self.stats["hits"] += 1
            return self.cache[key]  # Cache HIT
        
        # Step 2: Cache MISS → read from DB
        self.stats["misses"] += 1
        value = self.db.get(key)
        if value:
            self.cache[key] = value  # Step 3: Store in cache
        return value
    
    def write(self, key, value):
        self.db[key] = value       # Write to DB
        self.cache.pop(key, None)  # Invalidate cache (NOT update!)

ca = CacheAside()
print("Read user:1 (miss):", ca.read("user:1"))
print("Read user:1 (hit): ", ca.read("user:1"))
ca.write("user:1", "Alice Updated")
print("After write, read (miss):", ca.read("user:1"))
print(f"Stats: {ca.stats}")

print("\n💡 Why delete instead of update? Avoids race conditions.")
print("   Two concurrent writes could put stale data in cache.")

In [ ]:
# ============================================================
# 2. All 4 Caching Strategies Compared
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════════╗
║  CACHING STRATEGIES COMPARISON                                    ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Strategy        │ Read path        │ Write path       │ Best for ║
║  ────────────────│──────────────────│──────────────────│──────────║
║  Cache-Aside     │ App checks cache │ App writes DB,   │ General  ║
║                  │ miss → DB+cache  │ deletes cache    │ purpose  ║
║                  │                  │                  │          ║
║  Read-Through    │ Cache auto-loads │ Same as c-a but  │ Read     ║
║                  │ from DB on miss  │ cache owns load  │ heavy    ║
║                  │                  │                  │          ║
║  Write-Through   │ Same as above    │ Write cache+DB   │ Strong   ║
║                  │                  │ synchronously    │ consist. ║
║                  │                  │                  │          ║
║  Write-Behind    │ Same as above    │ Write cache,     │ Write    ║
║  (Write-Back)    │                  │ async flush DB   │ heavy    ║
║                  │                  │                  │          ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Trade-offs:                                                       ║
║  • Write-Through: slower writes (2 writes), but data consistent  ║
║  • Write-Behind: fast writes, but risk data loss (cache crash)    ║
║  • Cache-Aside: most flexible + most popular in production        ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 3. Redis vs Memcached
# ============================================================

print("""
REDIS vs MEMCACHED
══════════════════

┌───────────────────┬─────────────────────┬─────────────────────┐
│ Feature           │ Redis               │ Memcached           │
├───────────────────┼─────────────────────┼─────────────────────┤
│ Data structures   │ Strings, Lists,     │ Strings only        │
│                   │ Sets, Hashes,       │                     │
│                   │ Sorted Sets, Streams│                     │
│ Persistence       │ RDB + AOF           │ None (pure cache)   │
│ Replication       │ Master-Replica      │ No native           │
│ Pub/Sub           │ Yes                 │ No                  │
│ Lua scripting     │ Yes                 │ No                  │
│ Max value size    │ 512 MB              │ 1 MB                │
│ Multi-threading   │ Single-threaded *   │ Multi-threaded      │
│ Memory efficiency │ Higher overhead     │ More efficient      │
│ Use case          │ Cache + data store  │ Pure caching        │
└───────────────────┴─────────────────────┴─────────────────────┘

* Redis 6+ has I/O threading, but command execution is single-threaded

💡 Interview answer:
  "Redis for 95% of cases — richer data structures, persistence,
   pub/sub. Memcached only if you need pure multi-threaded caching
   with minimal memory overhead."
""")

In [ ]:
# ============================================================
# 4. Eviction Policies
# ============================================================

from collections import OrderedDict

class LRUCache:
    """LRU = Least Recently Used — evict the key not accessed the longest."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = OrderedDict()
    
    def get(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)  # Mark as recently used
            return self.cache[key]
        return -1
    
    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            evicted = self.cache.popitem(last=False)  # Remove oldest
            print(f"  Evicted: {evicted}")

lru = LRUCache(3)
for k, v in [("A", 1), ("B", 2), ("C", 3)]:
    lru.put(k, v)
print(f"Cache: {list(lru.cache.items())}")

lru.get("A")  # A is now most recently used
lru.put("D", 4)  # B should be evicted (least recently used)
print(f"After get(A)+put(D): {list(lru.cache.items())}")

print("""
Eviction Policies Summary:
  LRU  — Evict least recently used (most common)
  LFU  — Evict least frequently used (count-based)  
  FIFO — Evict first inserted
  TTL  — Evict after time expires
  Random — Random eviction (surprisingly decent!)

  Redis default: noeviction (return error when full)
  Redis recommended: allkeys-lru or volatile-lru
""")

In [ ]:
# ============================================================
# 5. Cache Stampede & Thundering Herd
# ============================================================

print("""
CACHE STAMPEDE (aka Thundering Herd)
════════════════════════════════════

Problem:
  1. Hot key expires in cache
  2. 1000 concurrent requests see cache miss
  3. All 1000 hit the DB simultaneously
  4. DB crashes under load 💥

Solutions:

1️⃣ LOCKING (Mutex)
   First request gets lock → fetches from DB → fills cache
   Other requests wait or get stale data

2️⃣ EARLY EXPIRATION (Probabilistic)
   Before TTL expires, randomly refresh in background
   gap_time = TTL - current_time
   if random() < exp(-gap_time * beta):
       refresh_cache_async()

3️⃣ STALE-WHILE-REVALIDATE
   Return stale data while refreshing in background
   (HTTP Cache-Control: stale-while-revalidate=60)

4️⃣ NEVER EXPIRE + ASYNC REFRESH
   Cache never expires (TTL=∞)
   Background job refreshes periodically
   Best for: config data, popular feeds


CACHE PENETRATION (different problem!):
  Requests for keys that DON'T EXIST in DB
  → Always misses cache AND DB
  Fix: Cache null values with short TTL
       Or use Bloom Filter to check existence first
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Cache-aside vs write-through? | Cache-aside: app manages cache (lazy). Write-through: cache+DB updated synchronously |
| 2 | Why delete cache on write, not update? | Avoids race condition between two concurrent writers storing stale data |
| 3 | What is cache stampede? Fix? | Hot key expires → all requests hit DB. Fix: locking, early refresh, stale-while-revalidate |
| 4 | Redis vs Memcached? | Redis: rich data types, persistence, pub/sub. Memcached: simpler, multi-threaded, less memory |
| 5 | How do you handle cache invalidation? | TTL for auto-expire, event-driven invalidation, pub/sub notifications, versioned keys |
| 6 | Cache penetration vs stampede? | Penetration = non-existent keys. Stampede = popular key expires. Different solutions |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Cache-Aside is king (most flexible, most used)        │
## │  • Delete cache on write, NEVER update directly          │
## │  • Redis for 95% of cases (rich types + persistence)     │
## │  • LRU is the default eviction policy                    │
## │  • Stampede: lock or early refresh or stale-while-reval  │
## │  • Cache 80/20 — 20% of data serves 80% of requests     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Database Choices — SQL vs NoSQL Decision Framework**